In [1]:
# Timestamp,ZoneID,WindSpeed_kn,Weather,WindDirection,Season,Sensitivity,PopupCount,Distance_m
# 2025-09-07T10:00:00Z,Z01,3.2,clear,N,autumn,4,0,20
# 2025-09-07T10:05:00Z,Z01,22.1,clear,NE,autumn,5,3,20
# 2025-09-07T10:10:00Z,Z01,27.8,rain,E,autumn,6,8,20
# ...


In [ ]:
import os
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any

# sklearn 전처리/파이프라인
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import joblib

# Keras (TensorFlow)
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# -----------------------------
# 0) 사용자 설정
# -----------------------------
INPUT_CSV = "maintenance_logs.csv"  # 아래 스키마 참고
MODEL_DIR = "ae_artifacts"
os.makedirs(MODEL_DIR, exist_ok=True)

# 어떤 컬럼을 사용할지(최소 구성)
CAT_COLS = ["Weather"]                  # 선택: ["ZoneID","Weather","WindDirection","Season"]
NUM_COLS = ["WindSpeed_kn", "Sensitivity", "PopupCount"]  # 선택: +["Distance_m"]
TIME_COL  = "Timestamp"                 # 있으면 유지, 없어도 무방

# 임계치 분위수(정상 데이터 분포에서 자동 산출)
THRESH_PCTL = 99.0

# 과민/둔감 서브타입 규칙 (간단히 조정 가능)
HIGH_WIND_KN = 20.0       # 강풍 기준
LOW_WIND_KN  = 8.0        # 약풍 기준
HIGH_POPUP   = 6.0        # 윈도우 당 팝업 많음 기준(데이터 스케일에 맞춰 조정)
LOW_POPUP    = 0.5        # 윈도우 당 팝업 거의 없음 기준

# -----------------------------
# 1) 데이터 로드 & 전처리 파이프라인
# -----------------------------
def load_data(path: str) -> pd.DataFrame:
    """
    maintenance_logs.csv 스키마 (예시):
      - Timestamp: 시각(ISO 또는 숫자)        [선택]
      - ZoneID: 구역 ID                       [선택]
      - WindSpeed_kn: 풍속(노트)              [필수]
      - Weather: 날씨 문자열(clear/rain/...)  [필수]
      - WindDirection: 8방(N,NE,...)          [선택]
      - Season: 계절                          [선택]
      - Sensitivity: 적용 민감도(정수 1~9)    [필수]
      - PopupCount: 시간창 내 팝업 횟수       [필수]
      - Distance_m: 센서-풍속계 거리          [선택]
    """
    df = pd.read_csv(path)
    # 필수 컬럼 체크
    req = set(CAT_COLS + NUM_COLS)
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}\n현재 컬럼: {df.columns.tolist()}")
    # 타입 보정
    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=NUM_COLS + CAT_COLS).reset_index(drop=True)
    return df

def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
            ("num", MinMaxScaler(), NUM_COLS)
        ],
        remainder="drop"
    )

# -----------------------------
# 2) 오토인코더 모델 정의
# -----------------------------
def build_autoencoder(input_dim: int) -> keras.Model:
    inp = keras.Input(shape=(input_dim,))
    # 인코더
    x = layers.Dense(3, activation="relu")(inp)
    z = layers.Dense(2, activation="relu")(x)     # 잠재(2D)
    # 디코더
    x = layers.Dense(3, activation="relu")(z)
    out = layers.Dense(input_dim, activation="sigmoid")(x)  # 입력을 0~1 스케일로 맞출 예정

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer="adam", loss="mse")
    return model

# -----------------------------
# 3) 학습 + 임계치 산출
# -----------------------------
def train_and_threshold(df: pd.DataFrame) -> Dict[str, Any]:
    """
    정상 데이터만으로 학습한다는 가정.
    초기에는 정상/이상 구분이 없을 수 있으므로, 전 기간 중간값 근처 구간을 '정상 후보'로 쓰거나
    운영 상 '정상으로 확신되는 기간'의 로그만 추려서 넣는 걸 권장.
    여기서는 단순히 전 데이터를 사용(실전에서는 정상 기간만 선택 권장).
    """
    preproc = build_preprocessor()
    X = df[CAT_COLS + NUM_COLS]
    X_train, X_val = train_test_split(X, test_size=0.2, random_state=42, shuffle=True)

    # 전처리 적합 & 변환 -> 0~1 스케일 + 원핫
    Xt_train = preproc.fit_transform(X_train)
    Xt_val   = preproc.transform(X_val)

    input_dim = Xt_train.shape[1]
    ae = build_autoencoder(input_dim)

    es = callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")
    hist = ae.fit(
        Xt_train, Xt_train,
        validation_data=(Xt_val, Xt_val),
        epochs=100,
        batch_size=128,
        verbose=1,
        callbacks=[es]
    )

    # 복원오차 분포 → 임계치
    recon_val = ae.predict(Xt_val, verbose=0)
    mse_val = np.mean(np.square(Xt_val - recon_val), axis=1)
    threshold = float(np.percentile(mse_val, THRESH_PCTL))

    # 저장(모델, 전처리, 임계치)
    ae.save(os.path.join(MODEL_DIR, "ae_model.keras"))
    joblib.dump(preproc, os.path.join(MODEL_DIR, "preprocessor.joblib"))
    with open(os.path.join(MODEL_DIR, "threshold.txt"), "w") as f:
        f.write(str(threshold))

    return {
        "input_dim": input_dim,
        "val_loss_min": float(np.min(hist.history["val_loss"])),
        "threshold": threshold,
        "pctl": THRESH_PCTL,
        "n_train": int(Xt_train.shape[0]),
        "n_val": int(Xt_val.shape[0])
    }

# -----------------------------
# 4) 추론 + 서브타입(과민/둔감) 분류
# -----------------------------
def load_artifacts():
    ae = keras.models.load_model(os.path.join(MODEL_DIR, "ae_model.keras"))
    preproc = joblib.load(os.path.join(MODEL_DIR, "preprocessor.joblib"))
    with open(os.path.join(MODEL_DIR, "threshold.txt"), "r") as f:
        threshold = float(f.read().strip())
    return ae, preproc, threshold

def classify_subtype(row: pd.Series) -> str:
    """
    간단 규칙:
      - 강풍(>=20kn) & 팝업 거의 없음(<=0.5) → 'under-sensitive'(둔감/고장 의심)
      - 약풍(<8kn)  & 팝업 많음(>=6)         → 'over-sensitive'(과민/고장 의심)
      - 그 외 → 'other'
    필요시 바람 계절/구역별 보정 가능.
    """
    ws = float(row["WindSpeed_kn"])
    pc = float(row["PopupCount"])
    if ws >= HIGH_WIND_KN and pc <= LOW_POPUP:
        return "under-sensitive"
    if ws < LOW_WIND_KN and pc >= HIGH_POPUP:
        return "over-sensitive"
    return "other"

def predict_anomaly(df_new: pd.DataFrame) -> pd.DataFrame:
    ae, preproc, threshold = load_artifacts()

    df_in = df_new.dropna(subset=CAT_COLS + NUM_COLS).copy()
    Xt = preproc.transform(df_in[CAT_COLS + NUM_COLS])

    recon = ae.predict(Xt, verbose=0)
    mse = np.mean(np.square(Xt - recon), axis=1)
    df_in["ReconMSE"] = mse
    df_in["Anomaly"] = df_in["ReconMSE"] > threshold

    # 서브타입 라벨(과민/둔감/기타)
    df_in["Subtype"] = df_in.apply(classify_subtype, axis=1)
    # 이상 중에서도 서브타입을 표시 (정상이라면 'normal')
    df_in["AnomalyTag"] = np.where(
        df_in["Anomaly"],
        df_in["Subtype"],
        "normal"
    )
    return df_in.sort_values("ReconMSE", ascending=False)

# -----------------------------
# 5) 실행 예시
# -----------------------------
if __name__ == "__main__":
    # 5-1) 학습
    df_logs = load_data(INPUT_CSV)
    summary = train_and_threshold(df_logs)
    print("=== TRAIN SUMMARY ===")
    for k, v in summary.items():
        print(f"{k}: {v}")

    # 5-2) 추론 데모 (최근 20건)
    demo_df = df_logs.tail(20).copy()
    pred = predict_anomaly(demo_df)
    print("\n=== PREDICTION (last 20) ===")
    print(pred[["WindSpeed_kn","Weather","Sensitivity","PopupCount","ReconMSE","Anomaly","AnomalyTag"]].head(20).to_string(index=False))
